# Workshop 3 — EDA, Cleaning & Schema Harmonization
**Part A · Steps 1-2 — World Happiness 2015-2019**

This notebook profiles the five yearly CSVs, documents the data-quality
findings, proposes a unified analytical schema and validates it with
**Great Expectations**.

The reusable implementation lives in `src/harmonize.py` (`build_unified()`);
this notebook only *drives* it, so the Airflow DAG runs exactly the same
code path (single implementation).

In [1]:
import sys
from pathlib import Path

REPO = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(REPO))

import pandas as pd

from src import config
from src.harmonize import RENAME, UNIFIED_COLUMNS, build_unified, load_aliases

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

RAW_FILES = sorted(config.RAW_DIR.glob("20*.csv"))
{p.stem: p for p in RAW_FILES}

{'2015': PosixPath('/home/drz/Documents/semesterV/etl/Activities/Workshop/3/data/raw/2015.csv'),
 '2016': PosixPath('/home/drz/Documents/semesterV/etl/Activities/Workshop/3/data/raw/2016.csv'),
 '2017': PosixPath('/home/drz/Documents/semesterV/etl/Activities/Workshop/3/data/raw/2017.csv'),
 '2018': PosixPath('/home/drz/Documents/semesterV/etl/Activities/Workshop/3/data/raw/2018.csv'),
 '2019': PosixPath('/home/drz/Documents/semesterV/etl/Activities/Workshop/3/data/raw/2019.csv')}

## 1. Raw profiling — one table per year (data-quality observations)

Schema, dtypes, nulls and duplicates are inspected **without modifying**
the source files (they stay untouched in `data/raw/`).

In [2]:
raw = {int(p.stem): pd.read_csv(p) for p in RAW_FILES}

rows = []
for year, df in sorted(raw.items()):
    rows.append(
        {
            "year": year,
            "rows": df.shape[0],
            "cols": df.shape[1],
            "null_cells": int(df.isna().sum().sum()),
            "dup_rows": int(df.duplicated().sum()),
            "numeric_cols": int(df.select_dtypes("number").shape[1]),
        }
    )
quality = pd.DataFrame(rows)
quality.loc[quality.index[-1], ""] = None
print(quality.to_string(index=False))
print("TOTAL rows:", int(quality["rows"].sum()))

 year  rows  cols  null_cells  dup_rows  numeric_cols     
 2015   158    12           0         0            10  NaN
 2016   157    13           0         0            11  NaN
 2017   155    12           0         0            11  NaN
 2018   156     9           1         0             8  NaN
 2019   156     9           0         0             8 None
TOTAL rows: 782


In [3]:
nulls_by_year = pd.DataFrame(
    {year: df.isna().sum() for year, df in sorted(raw.items())}
).T
print("nulls per column (only non-zero shown):")
print(nulls_by_year.loc[:, nulls_by_year.sum() > 0].to_string())

nulls per column (only non-zero shown):
      Perceptions of corruption
2015                        NaN
2016                        NaN
2017                        NaN
2018                        1.0
2019                        0.0


**Finding:** exactly **one** missing cell in the whole dataset —
`corruption` for *United Arab Emirates* in 2018. Everything else is
complete, with 0 duplicated rows per file.

In [4]:
COUNTRY_COL = {2015: "Country", 2016: "Country", 2017: "Country",
               2018: "Country or region", 2019: "Country or region"}
dups = {year: int(df.duplicated(subset=[COUNTRY_COL[year]]).sum())
        for year, df in sorted(raw.items())}
print("duplicate country rows per year:", dups)

duplicate country rows per year: {2015: 0, 2016: 0, 2017: 0, 2018: 0, 2019: 0}


## 2. Schema differences between years

The same concept changes its column name every year — this is the core
integration problem of the workshop.

In [5]:
schema = pd.DataFrame(
    {
        year: {col: "✓" for col in df.columns}
        for year, df in sorted(raw.items())
    }
).fillna("")
schema["unified_name"] = [
    next(
        (RENAME[y].get(col) for y in sorted(RENAME) if RENAME[y].get(col)),
        "",
    )
    if any(RENAME[y].get(col) for y in RENAME)
    else "— dropped —"
    for col in schema.index
]
print(schema.to_string())

                              2015 2016 2017 2018 2019     unified_name
Country                          ✓    ✓    ✓                    country
Region                           ✓    ✓                          region
Happiness Rank                   ✓    ✓                     — dropped —
Happiness Score                  ✓    ✓                 happiness_score
Standard Error                   ✓                          — dropped —
Economy (GDP per Capita)         ✓    ✓                             gdp
Family                           ✓    ✓    ✓                     family
Health (Life Expectancy)         ✓    ✓                          health
Freedom                          ✓    ✓    ✓                    freedom
Trust (Government Corruption)    ✓    ✓                      corruption
Generosity                       ✓    ✓    ✓    ✓    ✓       generosity
Dystopia Residual                ✓    ✓                     — dropped —
Lower Confidence Interval             ✓                     — dr

### Unified schema proposal

| unified column | source |
|---|---|
| `country` | `Country` / `Country or region` |
| `year` | **file name** (no year column exists in the files) |
| `region` | `Region` (2015-2016 only) → propagated |
| `gdp` | `Economy (GDP per Capita)` / `GDP per capita` / ... |
| `family` | `Family` / `Social support` |
| `health` | `Health (Life Expectancy)` / `Healthy life expectancy` |
| `freedom` | `Freedom` / `Freedom to make life choices` |
| `generosity` | `Generosity` |
| `corruption` | `Trust (Government Corruption)` / `Perceptions of corruption` |
| `happiness_score` | `Happiness Score` / `Happiness.Score` / `Score` |

**Dropped on purpose (target leakage / derived):** `Happiness Rank`,
`Standard Error`, confidence intervals, `Whisker.*`, `Dystopia Residual`.

## 3. Inconsistent country names

~170 raw names but fewer than 141 countries appear in all five years: part
of the difference is *the same country written differently*. A versioned
alias dictionary is applied **before** the merge.

In [6]:
raw_names = sorted(
    {
        str(v).strip()
        for year, df in sorted(raw.items())
        for v in df["Country" if year in (2015, 2016, 2017) else "Country or region"]
    }
)
print(f"distinct raw country names: {len(raw_names)}")

aliases = pd.read_csv(config.REFERENCE_DIR / "country_aliases.csv")
print("\ncountry_aliases.csv (variant -> canonical):")
print(aliases.to_string(index=False))

distinct raw country names: 170

country_aliases.csv (variant -> canonical):
                 variant           canonical
       Trinidad & Tobago Trinidad and Tobago
 Hong Kong S.A.R., China           Hong Kong
Taiwan Province of China              Taiwan
               Macedonia     North Macedonia
         Northern Cyprus        North Cyprus
       Somaliland Region   Somaliland region


In [7]:
# Any variant still present after harmonization would break dim_country
# and the "predictions by country" KPI -> verify explicitly.
df = build_unified(write=False)
per_year = df.groupby("country")["year"].nunique()
incomplete = per_year[per_year < 5]
print(f"\ncanonical countries: {df['country'].nunique()}")
print(f"present in all 5 years: {(per_year == 5).sum()}")
print(f"in <5 years: {len(incomplete)} (gaps in the SOURCE, not name variants):")
print(incomplete.to_string())


canonical countries: 164
present in all 5 years: 146
in <5 years: 18 (gaps in the SOURCE, not name variants):
country
Angola                      4
Belize                      3
Central African Republic    4
Comoros                     3
Djibouti                    1
Gambia                      1
Laos                        4
Lesotho                     4
Mozambique                  4
Namibia                     4
Oman                        1
Puerto Rico                 1
Somalia                     4
Somaliland region           2
South Sudan                 4
Sudan                       4
Suriname                    2
Swaziland                   2


## 4. Harmonization: types, missing values, duplicates

Cleaning decisions (justified in the README):

| problem | decision | justification |
|---|---|---|
| 1 null (`corruption`, UAE 2018) | impute with UAE's 2017+2019 mean | dropping the row loses a whole country-year; 5 other indicators are valid |
| exact zeros (Sierra Leone, ...) | **keep** | published values at the bottom of the scale, not disguised nulls |
| `generosity` ≈ 0.838 (2017) | **keep** (flagged by IQR) | real value of a real country; goal is the pipeline, not model tuning |
| schema drift | `RENAME` map per year | one unified schema |
| region only 2015-16 | propagate, fallback `Unknown` | region is not a feature (not in the event contract) |

In [8]:
unified = build_unified(write=True)

print("shape:", unified.shape)
print("dtypes:")
print(unified.dtypes.to_string())
print("nulls:", int(unified.isna().sum().sum()))
print("duplicate (country, year):", int(unified.duplicated(["country", "year"]).sum()))
print("\nrows per year:")
print(unified["year"].value_counts().sort_index().to_string())
print("\nhead:")
print(unified.head(10).to_string(index=False))

shape: (782, 10)
dtypes:
country             string
year                 int64
region              string
gdp                float64
family             float64
health             float64
freedom            float64
generosity         float64
corruption         float64
happiness_score    float64
nulls: 0
duplicate (country, year): 0

rows per year:
year
2015    158
2016    157
2017    155
2018    156
2019    156

head:
    country  year                    region     gdp  family  health  freedom  generosity  corruption  happiness_score
Switzerland  2015            Western Europe 1.39651 1.34951 0.94143  0.66557     0.29678     0.41978            7.587
    Iceland  2015            Western Europe 1.30232 1.40223 0.94784  0.62877     0.43630     0.14145            7.561
    Denmark  2015            Western Europe 1.32548 1.36058 0.87464  0.64938     0.34139     0.48357            7.527
     Norway  2015            Western Europe 1.45900 1.33095 0.88521  0.66973     0.34699     0.36503       

In [9]:
# The single imputed value (surface it from the harmonizer for the report)
imputations = getattr(build_unified, "last_imputations", [])
print("imputed values:")
for note in imputations:
    print(" -", note)

imputed values:
 - corruption [United Arab Emirates 2018] was null -> 0.3120 (mean of United Arab Emirates other years)


## 5. Outliers — reported, never deleted

IQR rule per feature: values outside `[Q1 - 1.5·IQR, Q3 + 1.5·IQR]` are
counted (and plotted), but **kept**: they are genuine published data.

In [10]:
features = ["gdp", "family", "health", "freedom", "generosity", "corruption", "happiness_score"]
iqr_rows = []
for col in features:
    q1, q3 = unified[col].quantile([0.25, 0.75])
    low, high = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
    out = unified[(unified[col] < low) | (unified[col] > high)]
    iqr_rows.append({"column": col, "low": round(low, 3), "high": round(high, 3),
                     "n_outliers": len(out),
                     "examples": ", ".join(out.nlargest(2, col)["country"])})
print(pd.DataFrame(iqr_rows).to_string(index=False))

         column    low  high  n_outliers         examples
            gdp -0.338 2.181           0                 
         family  0.183 2.014          10    Syria, Malawi
         health -0.112 1.360           0                 
        freedom -0.022 0.863           0                 
     generosity -0.093 0.502          17 Myanmar, Myanmar
     corruption -0.099 0.309          68    Rwanda, Qatar
happiness_score  1.990 8.709           0                 


In [11]:
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for ax, col in zip(axes.flat, features):
    sns.boxplot(y=unified[col], ax=ax, color="steelblue")
    ax.set_title(col)
axes.flat[-1].axis("off")
fig.suptitle("Boxplots — outliers are reported, never removed")
fig.tight_layout()
plt.show()

/tmp/nix-shell.e6jU9N/ipykernel_52482/3111760717.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Distributions & relationships (feature engineering input)

In [12]:
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for ax, col in zip(axes.flat, features):
    sns.histplot(unified[col], ax=ax, kde=True, color="seagreen")
    ax.set_title(col)
axes.flat[-1].axis("off")
fig.suptitle("Feature distributions (unified dataset)")
fig.tight_layout()
plt.show()

/tmp/nix-shell.e6jU9N/ipykernel_52482/3279628200.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [13]:
corr = unified[features].corr()["happiness_score"].sort_values(ascending=False)
print("correlation with happiness_score:")
print(corr.round(2).to_string())

fig, ax = plt.subplots(figsize=(8, 4))
corr.drop("happiness_score").plot.bar(ax=ax, color="steelblue")
ax.set_ylabel("Pearson r")
ax.set_title("Correlation of each feature with the target")
plt.tight_layout()
plt.show()

correlation with happiness_score:
happiness_score    1.00
gdp                0.79
health             0.74
family             0.65
freedom            0.55
corruption         0.40
generosity         0.14


/tmp/nix-shell.e6jU9N/ipykernel_52482/3581859890.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(unified[features].corr(), annot=True, fmt=".2f", cmap="vlag",
            center=0, ax=ax)
ax.set_title("Feature correlation matrix")
plt.tight_layout()
plt.show()

/tmp/nix-shell.e6jU9N/ipykernel_52482/1547196395.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [15]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["gdp", "health", "family"]):
    sns.scatterplot(data=unified, x=col, y="happiness_score", hue="year",
                    palette="viridis", ax=ax, legend=False)
    ax.set_title(f"happiness_score vs {col}")
plt.tight_layout()
plt.show()

/tmp/nix-shell.e6jU9N/ipykernel_52482/1956873616.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [16]:
print("region coverage (propagated from 2015-2016):")
print(unified["region"].value_counts().head(15).to_string())

region coverage (propagated from 2015-2016):
region
Sub-Saharan Africa                 195
Central and Eastern Europe         145
Latin America and Caribbean        111
Western Europe                     105
Middle East and Northern Africa     96
Southeastern Asia                   44
Southern Asia                       35
Eastern Asia                        30
North America                       10
Australia and New Zealand           10
Unknown                              1


## 7. Great Expectations — data-quality gate on the unified dataset

29 expectations: column order, no nulls, year domain, per-feature ranges,
`(country, year)` uniqueness and dtypes. The Airflow DAG runs this gate
**before** training and stops if it fails.

In [17]:
from src.validate import run_validation

summary = run_validation(unified, raise_on_failure=False)
print(f"rows validated : {summary['rows_validated']}")
print(f"expectations   : {summary['n_expectations']}")
print(f"failed         : {summary['n_failed']}")
print("result         :", "PASSED ✓" if summary["successful"] else "FAILED ✗")
if summary["failed"]:
    for failure in summary["failed"]:
        print("  FAILED", failure)

Calculating Metrics:   0%|          | 0/94 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/94 [00:00<?, ?it/s]

Calculating Metrics:   2%|▏         | 2/94 [00:00<00:00, 3371.63it/s]

Calculating Metrics:   2%|▏         | 2/94 [00:00<00:00, 1787.85it/s]

Calculating Metrics:   3%|▎         | 3/94 [00:00<00:00, 1950.54it/s]

Calculating Metrics:   3%|▎         | 3/94 [00:00<00:00, 1482.26it/s]

... (3 outputs left out)

Calculating Metrics: 100%|██████████| 94/94 [00:00<00:00, 4261.44it/s]

Calculating Metrics: 100%|██████████| 94/94 [00:00<00:00, 4209.40it/s]

Calculating Metrics: 100%|██████████| 94/94 [00:00<00:00, 4156.10it/s]

rows validated : 782
expectations   : 29
failed         : 0
result         : PASSED ✓


## 8. Deliverables produced by this notebook

* `data/processed/unified_happiness.parquet` (+ `.csv`) — 782 rows,
  0 nulls, unique `(country, year)`, validated.
* Data-quality observations (sections 1-6) and the unified schema proposal.

Next: `notebooks/model_training.ipynb` (features + model).

In [18]:
print(f"wrote {config.UNIFIED_PARQUET.relative_to(config.ROOT)}")
print(f"wrote {config.UNIFIED_CSV.relative_to(config.ROOT)}")
unified.head(10)

wrote data/processed/unified_happiness.parquet
wrote data/processed/unified_happiness.csv


,country,year,region,gdp,family,health,freedom,generosity,corruption,happiness_score
0,Switzerland,2015,Western Europe,1.39651,1.34951,0.94143,0.66557,0.29678,0.41978,7.587
1,Iceland,2015,Western Europe,1.30232,1.40223,0.94784,0.62877,0.43630,0.14145,7.561
2,Denmark,2015,Western Europe,1.32548,1.36058,0.87464,0.64938,0.34139,0.48357,7.527
3,Norway,2015,Western Europe,1.45900,1.33095,0.88521,0.66973,0.34699,0.36503,7.522
4,Canada,2015,North America,1.32629,1.32261,0.90563,0.63297,0.45811,0.32957,7.427
5,Finland,2015,Western Europe,1.29025,1.31826,0.88911,0.64169,0.23351,0.41372,7.406
6,Netherlands,2015,Western Europe,1.32944,1.28017,0.89284,0.61576,0.47610,0.31814,7.378
7,Sweden,2015,Western Europe,1.33171,1.28907,0.91087,0.65980,0.36262,0.43844,7.364
8,New Zealand,2015,Australia and New Zealand,1.25018,1.31967,0.90837,0.63938,0.47501,0.42922,7.286
9,Australia,2015,Australia and New Zealand,1.33358,1.30923,0.93156,0.65124,0.43562,0.35637,7.284
